# NB03 — Grading SemCat: found / missed / wrong

**Goal.** Adjudicate Q2 by joining NB01's frozen SemCat-only answer to NB02's independent
ground truth. Produce per-facet precision and recall, a per-table found/missed/wrong verdict,
and the H0 test (does SemCat beat a name regex?).

**Discriminating query (run first).** The pre-registered stratified 200-column audit:
100 columns SemCat labelled (→ **precision**) and 100 ground-truth annotation/coordinate
columns regardless of label (→ **recall**). Per `RESEARCH_PLAN.md` this is *the* H2-vs-H3 test,
since H2 and H3 differ only on whether the labels SemCat does apply can be trusted.

**Adjudication is on values, not names.** Judging a `pfam_id` label by whether the column is
*called* something Pfam-ish would re-import the assumption under test. `src/validators.py`
checks sampled values against the format each label claims (`pfam_id` → `PF#####`, `strand` →
`+`/`-`, `go_term` → `GO:#######`, …).

**One class of label is scored apart.** `identifier`, `annotation_text`, `accession`, `name`,
`label`, `category` assert nothing a sample could refute. They are reported as **vacuous**
rather than correct — a label that cannot be wrong cannot be informative either.

**Failures are separated into three kinds**, because a first pass showed that one "wrong"
bucket conflated things with very different implications:

| verdict | meaning | whose defect |
|---|---|---|
| `wrong_label` | the values are not that vocabulary at all (`cog_id` holding `J`, `A`, `K`) | SemCat's |
| `wrong_cardinality` | right vocabulary, but the column is a delimited list (`go_term` holding `GO:0000287;GO:0008927`) | SemCat's, but milder and different — still consequential, since `WHERE go = 'GO:0006064'` silently matches nothing |
| `sentinel_only` | every sampled value is a placeholder (`-`, `nan`, `None`) | **the audit's** — the label may be right for populated rows and a `LIMIT` sample simply missed them. Excluded from the denominator rather than counted against SemCat. |

Because the H2/H3 boundary sits at precision 0.80, precision is reported **twice**: strict
(cardinality errors count against SemCat) and lenient (they do not). If the two straddle the
threshold, the honest report is that this sample does not resolve H2 vs H3 — which the
pre-registered decision rule already provides for under "mixed evidence".

In [1]:
import sys, os, re, json, random, math, time
_T0 = time.time()
sys.path.insert(0, os.path.abspath("../src"))
import pandas as pd
from berdl_notebook_utils.setup_spark_session import get_spark_session
from semcat import SEMCAT, ACCESSIBLE_TENANTS
from validators import validate2, PATTERNS, VACUOUS, LABEL_TO_FAMILY, FAMILY

spark = get_spark_session(app_name="acatlas_nb03")
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 70); pd.set_option("display.max_rows", 240)
SEED = 20261002   # fixed so the audit sample is reproducible
random.seed(SEED)

gt = pd.read_csv("../data/ground_truth_columns.csv", keep_default_na=False)
cand = pd.read_csv("../data/semcat_candidates.csv")
cen = pd.read_csv("../data/annotation_sources.csv")
nb01 = json.load(open("../data/nb01_summary.json"))
nb02 = json.load(open("../data/nb02_summary.json"))
# The one definition of "accessible" (src/semcat.py); NB02 checked it against the live listing.
TENANTS = sorted(ACCESSIBLE_TENANTS)
assert set(gt.tenant.unique()) <= set(TENANTS), set(gt.tenant.unique()) - set(TENANTS)
print("accessible tenants:", TENANTS)
print(f"ground-truth columns: {len(gt):,} in {gt.fq.nunique()} tables")

accessible tenants: ['bervodata', 'culturebotai', 'ese', 'globalusers', 'kbase', 'kescience', 'mamillerpa', 'my', 'nmdc', 'refdata']
ground-truth columns: 67,910 in 840 tables


## 1. The audited frame

Restricted to tables that are **both** live and indexed by SemCat. Anything stale (in SemCat,
gone from the lakehouse) or missing (live, unknown to SemCat) cannot be scored on label
quality — SemCat either has no column to label or no table to label it on. Those are coverage
failures, already measured in NB02 §6, and mixing them into precision/recall would conflate
two different defects.

In [2]:
sc_cols = spark.sql(f'''
SELECT concat(tenant,'.',database,'.',table_name) AS fq, column_name, type AS sc_type,
       content_type, role, concept_label, facet_layer, facet_confidence, ndv, null_fraction
FROM {SEMCAT}.columns
WHERE tenant IN ({",".join("'"+t+"'" for t in TENANTS)})
''').toPandas()
print(f"SemCat columns in accessible tenants: {len(sc_cols):,}")

# schema_sig lets us collapse cloned tables (e.g. NMDC dated staging copies carrying an
# identical genome_feature_set) to one observation, as the plan's unit-of-analysis section
# requires. Pulled from semantic_catalog.tables.
sig = spark.sql(f'''
SELECT concat(tenant,'.',database,'.',table_name) AS fq, schema_sig
FROM {SEMCAT}.tables''').toPandas()
SIG = dict(zip(sig.fq, sig.schema_sig))

live_tbls = set(gt.fq); sc_tbls = set(sc_cols.fq)
frame_tbls = live_tbls & sc_tbls
print(f"live tables {len(live_tbls)}, SemCat tables {len(sc_tbls)}, AUDITED FRAME {len(frame_tbls)}")

gtf = gt[gt.fq.isin(frame_tbls)].copy()
scf = sc_cols[sc_cols.fq.isin(frame_tbls)].copy()
gtf["node"] = gtf.fq + "." + gtf.column_name
scf["node"] = scf.fq + "." + scf.column_name
print(f"columns in frame: ground truth {len(gtf):,}, SemCat {len(scf):,}")
print(f"content_type populated on {scf.content_type.notna().sum():,} "
      f"({scf.content_type.notna().mean():.1%}) of in-frame SemCat columns")

SemCat columns in accessible tenants: 66,548


live tables 840, SemCat tables 679, AUDITED FRAME 655
columns in frame: ground truth 66,390, SemCat 66,374
content_type populated on 5,458 (8.2%) of in-frame SemCat columns


In [3]:
# Facet-tag the ground truth, using the same rules NB02 used (independent of SemCat).
ln = gtf.column_name.str.lower()
RX_CV = (r"pfam|interpro|^ipr$|^ipr_|_ipr$|tigrfam|panther|prosite|hamap|superfamily|gene3d|sfld|"
 r"pirsf|^ko$|^ko_|_ko$|^koid$|ko_term|kofam|kegg|^ec$|^ec_|_ec$|ec_number|enzyme_commission|"
 r"^go$|^go_|_go$|^goid$|go_term|go_id|goslim|^cog$|^cog_|_cog$|eggnog|ortholog|cazy|dbcan|"
 r"antismash|mibig|big_scape|^gcf|smcog|uniprot|swissprot|trembl|metacyc|rhea|reactome|"
 r"seed|subsystem|resfam|merops|tcdb|signature|^hmm|pathway|^module|ontology_terms|"
 r"protein_family|family_id|domain|motif|operon|regulon|^amr|virulence|specialty")
RX_SPAN2 = (r"^(start|end|stop|begin|beg)$|_(start|end|stop|begin|beg|pos)$|"
            r"^(start|end|stop|begin|beg)_|posfrom|posto|position_range")
RX_GEO2 = r"^lat$|^lon$|^long$|latitude|longitude|^lat_|_lat$|^lon_|_lon$|_long$|bbox|cent_lat|dec_lat|huc"
LIBRARY_STRANDEDNESS = {"is_stranded", "stranded_orientation"}
READING_FRAME = {"frame", "phase", "cds_phase"}

def gt_facet(col):
    c = col.lower()
    if c in LIBRARY_STRANDEDNESS or c in READING_FRAME: return None
    if c in ("strand", "orientation") or c.endswith("_strand"): return "strand"
    if re.search(RX_GEO2, c): return None
    if re.search(RX_SPAN2, c): return "span"
    if re.search(RX_CV, c):   return "function_id"
    return None

gtf["gt_facet"] = gtf.column_name.map(gt_facet)
truth = gtf[gtf.gt_facet.notna()]
print("GROUND-TRUTH annotation/coordinate columns in frame, by facet:")
print(truth.gt_facet.value_counts().to_string())
print(f"total: {len(truth):,} columns in {truth.fq.nunique()} tables")

GROUND-TRUTH annotation/coordinate columns in frame, by facet:
gt_facet
function_id    621
span            88
strand          14
total: 723 columns in 192 tables


## 2. Discriminating query — the stratified 200-column audit

Sampling is stratified by facet (and, inside the precision stratum, by label) so no single
large family dominates, and seeded so the draw is reproducible.

In [4]:
def sample_strat(df, keycol, n, seed=SEED):
    """Round-robin across strata so small strata are represented."""
    rng = random.Random(seed)
    groups = {k: rng.sample(list(v.index), len(v)) for k, v in df.groupby(keycol)}
    order, picked = sorted(groups), []
    while len(picked) < n and any(groups[k] for k in order):
        for k in order:
            if groups[k] and len(picked) < n:
                picked.append(groups[k].pop())
    return df.loc[picked]

# --- PRECISION stratum: columns SemCat labelled with a relevant content_type ---
scf_lab = scf[scf.content_type.notna()].copy()
scf_lab["fam"] = scf_lab.content_type.map(LABEL_TO_FAMILY)
relevant = scf_lab[scf_lab.fam.notna() | scf_lab.content_type.isin(VACUOUS)]
prec_pool = relevant.copy()
print(f"precision pool (labelled, relevant or vacuous): {len(prec_pool):,}")
prec_sample = sample_strat(prec_pool, "content_type", 100)
print(f"precision sample: {len(prec_sample)} columns, {prec_sample.content_type.nunique()} labels")
print(prec_sample.content_type.value_counts().to_string())

precision pool (labelled, relevant or vacuous): 2,736
precision sample: 100 columns, 50 labels
content_type
accession                 3
annotation_text           3
category                  3
cog_category              3
cog_id                    3
contig_id                 3
coordinate                3
ec_number                 3
functional_group_score    3
gene_cluster_id           3
gene_id                   2
gene_locus_id             2
genome_id                 2
go_term                   2
identifier                2
interpro_id               2
kegg_gene_id              2
kegg_id                   2
kegg_module               2
kegg_module_id            2
kegg_pathway              2
kegg_pathway_id           2
kegg_reaction             2
ko_term                   2
label                     2
length                    2
name                      2
ontology_term             2
ontology_term_id          2
ontology_term_label       2
organism_id               2
pfam_id                 

In [5]:
# --- RECALL stratum: ground-truth columns, regardless of what SemCat said ---
rec_sample = sample_strat(truth, "gt_facet", 100)
print(f"recall sample: {len(rec_sample)} columns")
print(rec_sample.gt_facet.value_counts().to_string())
print()
print("by tenant:")
print(rec_sample.tenant.value_counts().to_string())

recall sample: 100 columns
gt_facet
function_id    43
span           43
strand         14

by tenant:
tenant
refdata        26
globalusers    23
kbase          19
kescience      19
nmdc           13


In [6]:
# Sample actual VALUES for every audited column -- DETERMINISTICALLY.
#
# The first version of this cell used `SELECT col ... LIMIT n` with no ORDER BY. Spark and
# Iceberg give no row-order guarantee for that, so repeated full-chain re-runs adjudicated
# different values and one column migrated between `wrong_cardinality` and `wrong_label`,
# moving reported lenient precision by ~2 points (Correction 6 in REPORT.md). The seed fixes
# which COLUMNS are audited; it never fixed which VALUES within them.
#
# Three properties are needed, and only this combination supplies all three:
#   deterministic -- a total order on a pure function of the value (xxhash64), so the same
#                    multiset of values comes back on every execution.
#   diverse       -- DISTINCT. Ordering by hash alone groups identical values together, so a
#                    repeated-value column returns one value 60 times (measured: gene_id on
#                    pfam_annotation_gff returned the same id repeatedly). Ordering by the
#                    value itself is worse still: '-' sorts below every alphanumeric, so
#                    placeholder-heavy columns would return nothing but sentinels.
#   affordable    -- DISTINCT over a billion-row high-cardinality column costs ~101 s. Where
#                    SemCat reports high ndv, a deterministic hash-bucket prefilter cuts the
#                    aggregation input ~256x (measured ~67 s) while remaining a pure function
#                    of the value. Skipped for low-ndv columns, where bucketing could drop the
#                    distinct count below n.
NDV = {(r.fq, r.column_name): r.ndv for r in scf.itertuples()}
HIGH_NDV = 100_000
BUCKETS = 256

def sample_values(fq, col, n=60):
    ndv = NDV.get((fq, col))
    hi = ndv is not None and not pd.isna(ndv) and ndv > HIGH_NDV
    bucket = (f"AND pmod(abs(xxhash64(CAST(`{col}` AS STRING))), {BUCKETS}) = 0" if hi else "")
    sql = (f"SELECT DISTINCT `{col}` AS v FROM {fq} "
           f"WHERE `{col}` IS NOT NULL {bucket} "
           f"ORDER BY xxhash64(CAST(v AS STRING)) LIMIT {n}")
    try:
        return [r["v"] for r in spark.sql(sql).collect()]
    except Exception as e:
        return f"ERROR: {str(e)[:90]}"

print(f"sampling is deterministic: DISTINCT + ORDER BY xxhash64, "
      f"hash-bucketed where ndv > {HIGH_NDV:,}")

need = {(r.fq, r.column_name) for r in prec_sample.itertuples()} | \
       {(r.fq, r.column_name) for r in rec_sample.itertuples()}
print(f"sampling values for {len(need)} distinct columns ...")
VALS = {}
for i, (fq, col) in enumerate(sorted(need), 1):
    VALS[(fq, col)] = sample_values(fq, col)
    if i % 40 == 0: print(f"  {i}/{len(need)}")
errs = sum(1 for v in VALS.values() if isinstance(v, str))
empty = sum(1 for v in VALS.values()
            if not isinstance(v, str) and not any(x is not None and str(x).strip() != "" for x in v))
print(f"done; {errs} columns failed to query, {empty} returned no non-empty value "
      f"(precision and recall samples together; section 2a breaks these down)")

sampling is deterministic: DISTINCT + ORDER BY xxhash64, hash-bucketed where ndv > 100,000
sampling values for 193 distinct columns ...


  40/193


  80/193


  120/193


  160/193


done; 0 columns failed to query, 19 returned no non-empty value (precision and recall samples together; section 2a breaks these down)


### 2a. Precision — do SemCat's labels hold up against the data?

In [7]:
def wilson(k, n, z=1.96):
    """95% Wilson score interval for k successes in n trials."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def normal_ci(k, n, z=1.96):
    """95% normal-approximation (Wald) interval, for comparison."""
    p = k / n
    h = z * math.sqrt(p * (1 - p) / n)
    return (p - h, p + h)

rows = []
for r in prec_sample.itertuples():
    v = VALS.get((r.fq, r.column_name))
    if isinstance(v, str):
        verdict, detail = "no_data", v
    else:
        verdict, detail = validate2(r.content_type, v)
    rows.append(dict(fq=r.fq, column_name=r.column_name, content_type=r.content_type,
                     family=LABEL_TO_FAMILY.get(r.content_type), verdict=verdict, detail=detail,
                     sample=str(v)[:70] if not isinstance(v, str) else ""))
prec = pd.DataFrame(rows)
print(prec.verdict.value_counts().to_string())
print()
JUDGED = ["correct", "wrong_cardinality", "wrong_label"]
checkable = prec[prec.verdict.isin(JUDGED)]
n_corr  = int((checkable.verdict == "correct").sum())
n_card  = int((checkable.verdict == "wrong_cardinality").sum())
n_wrong = int((checkable.verdict == "wrong_label").sum())
precision_strict  = n_corr / len(checkable) if len(checkable) else float("nan")
precision_lenient = (n_corr + n_card) / len(checkable) if len(checkable) else float("nan")
print(f"judgeable labels         : {len(checkable)}")
print(f"  correct                : {n_corr}")
print(f"  wrong_cardinality      : {n_card}   (right vocabulary, delimited list)")
print(f"  wrong_label            : {n_wrong}   (not that vocabulary at all)")
print()
ci_s, ci_l = wilson(n_corr, len(checkable)), wilson(n_corr + n_card, len(checkable))
wald_s = normal_ci(n_corr, len(checkable))
print(f"  PRECISION, strict      : {n_corr}/{len(checkable)} = {precision_strict:.1%}   "
      f"95% Wilson [{ci_s[0]:.3f}, {ci_s[1]:.3f}]   (normal approx. [{wald_s[0]:.3f}, {wald_s[1]:.3f}])")
print(f"  PRECISION, lenient     : {n_corr+n_card}/{len(checkable)} = {precision_lenient:.1%}   "
      f"95% Wilson [{ci_l[0]:.3f}, {ci_l[1]:.3f}]")
print(f"  strict lower bound clears the 0.80 H2 threshold: {ci_s[0] >= 0.80}")
print()
print(f"vacuous (unfalsifiable)  : {(prec.verdict=='vacuous').sum()}  "
      f"= {(prec.verdict=='vacuous').mean():.0%} of the audited sample")
print(f"sentinel_only (AUDIT limit, excluded): {(prec.verdict=='sentinel_only').sum()}")
print(f"no validator defined     : {(prec.verdict=='no_pattern').sum()}")
# Reconcile the no_data verdicts with the sampling cell's counts. validate2 checks for a
# vacuous label before it looks at values, and recall never reads values, so not every empty
# sample becomes a no_data verdict.
EMPTY = {k for k, v in VALS.items()
         if not isinstance(v, str) and not any(x is not None and str(x).strip() != "" for x in v)}
FAILED = {k for k, v in VALS.items() if isinstance(v, str)}
prec_keys = {(r.fq, r.column_name) for r in prec_sample.itertuples()}
vac_keys = {(r.fq, r.column_name) for r in prec_sample.itertuples() if r.content_type in VACUOUS}
n_nodata = int((prec.verdict == "no_data").sum())
print(f"unsamplable (no_data)    : {n_nodata} = {len(FAILED & prec_keys)} query failures + "
      f"{len((EMPTY & prec_keys) - vac_keys)} empty samples on non-vacuous labels")
print(f"  of the {len(EMPTY)} empty samples: {len((EMPTY & prec_keys) - vac_keys)} -> no_data here, "
      f"{len(EMPTY & vac_keys)} on vacuous labels (scored vacuous), "
      f"{len(EMPTY - prec_keys)} in the recall stratum only (recall does not read values)")
precision = precision_strict  # strict figure drives the decision rule

verdict
correct              47
no_pattern           19
vacuous              17
wrong_label           7
no_data               5
wrong_cardinality     4
sentinel_only         1

judgeable labels         : 58
  correct                : 47
  wrong_cardinality      : 4   (right vocabulary, delimited list)
  wrong_label            : 7   (not that vocabulary at all)

  PRECISION, strict      : 47/58 = 81.0%   95% Wilson [0.691, 0.891]   (normal approx. [0.709, 0.911])
  PRECISION, lenient     : 51/58 = 87.9%   95% Wilson [0.771, 0.940]
  strict lower bound clears the 0.80 H2 threshold: False

vacuous (unfalsifiable)  : 17  = 17% of the audited sample
sentinel_only (AUDIT limit, excluded): 1
no validator defined     : 19
unsamplable (no_data)    : 5 = 0 query failures + 5 empty samples on non-vacuous labels
  of the 19 empty samples: 5 -> no_data here, 2 on vacuous labels (scored vacuous), 12 in the recall stratum only (recall does not read values)


In [8]:
for kind, head in [("wrong_label", "MISLABELLED -- values are not that vocabulary"),
                   ("wrong_cardinality", "MULTI-VALUED -- right vocabulary, delimited list"),
                   ("sentinel_only", "INCONCLUSIVE -- all sampled values are placeholders (audit limit)")]:
    w = prec[prec.verdict == kind]
    print(f"{head}  ({len(w)})")
    for r in w.itertuples():
        print(f"  {r.fq}.{r.column_name}")
        print(f"      SemCat says `{r.content_type}` -- {r.detail[:135]}")
    print()
print("precision (strict) by label family, with 95% Wilson intervals:")
fam_prec = {}
for fam, s in checkable.groupby("family").verdict:
    k, n = int((s == "correct").sum()), len(s)
    lo, hi = wilson(k, n)
    fam_prec[fam] = dict(correct=k, judged=n, rate=k / n, ci_low=lo, ci_high=hi)
    print(f"  {fam:12s} {k}/{n} = {k/n:.0%}   [{lo:.2f}, {hi:.2f}]")

MISLABELLED -- values are not that vocabulary  (7)
  kbase.nmdc_arkin.cog_hierarchy_flat.cog_id
      SemCat says `cog_id` -- 26/26 values are not COG####; e.g. ['G', 'L', 'B']
  refdata.interpro.parent_child_tree.interpro_name
      SemCat says `interpro_id` -- 60/60 values are not IPR######; e.g. ['VAV1 protein, second SH3 domain', 'PET domain', 'Interleukin-1 receptor type 1']
  refdata.tcdb.acc_to_tcid.uniprot_accession
      SemCat says `uniprot_accession` -- 14/60 values are not a UniProt accession; e.g. ['KAG7176625.1', 'KAF9273289.1', 'AIU34623.1']
  kescience.pdb.pdb_uniprot_mapping.pdb_end
      SemCat says `coordinate` -- 1/60 values are not a number; e.g. ['66A']
  kbase.nmdc_arkin.annotation_hierarchies_unified.term_id
      SemCat says `ontology_term_id` -- 1/60 values are not PREFIX:LOCALID; e.g. ['5.3.-.-']
  kescience.mgnify.genome_cog.cog_category
      SemCat says `cog_id` -- 24/24 values are not COG####; e.g. ['G', 'L', 'B']
  kescience.mgnify.species.n_contigs
    

### 2b. Recall — did SemCat label the columns that matter?

A ground-truth column counts as **found** only if SemCat gave it a label in the *right family*.
`annotation_text` on a strand column is not a hit: it is a vacuous label that happens to sit on
a relevant column, which is exactly the failure mode the specific/generic split in NB01 was
introduced to expose.

In [9]:
sc_by_node = scf.set_index("node")[["content_type"]].to_dict()["content_type"]
rows = []
for r in rec_sample.itertuples():
    node = f"{r.fq}.{r.column_name}"
    ct = sc_by_node.get(node)
    fam = LABEL_TO_FAMILY.get(ct) if isinstance(ct, str) else None
    if ct is None or (isinstance(ct, float) and pd.isna(ct)):
        status = "unlabelled"
    elif ct in VACUOUS:
        status = "vacuous_label"
    elif fam == r.gt_facet:
        status = "found"
    else:
        status = "wrong_family"
    rows.append(dict(fq=r.fq, column_name=r.column_name, gt_facet=r.gt_facet,
                     semcat_label=ct if isinstance(ct, str) else None,
                     semcat_family=fam, status=status))
rec = pd.DataFrame(rows)
print(rec.status.value_counts().to_string())
print()
recall = (rec.status == "found").mean()
print(f"RECALL (apt label in the right family): {(rec.status=='found').sum()}/{len(rec)} = {recall:.1%}")
print()
print("recall by facet, with 95% Wilson intervals (per column):")
fac_rec = {}
for fac, s in rec.groupby("gt_facet").status:
    k, n = int((s == "found").sum()), len(s)
    lo, hi = wilson(k, n)
    fac_rec[fac] = dict(found=k, n=n, rate=k / n, ci_low=lo, ci_high=hi)
    print(f"  {fac:12s} {k}/{n} = {k/n:.0%}   [{lo:.2f}, {hi:.2f}]")
print()
print("what SemCat said instead, on the columns it did not get right:")
miss = rec[rec.status != "found"]
print(miss.groupby(["gt_facet", "status"]).size().to_string())

status
unlabelled       45
found            30
wrong_family     14
vacuous_label    11

RECALL (apt label in the right family): 30/100 = 30.0%

recall by facet, with 95% Wilson intervals (per column):
  function_id  11/43 = 26%   [0.15, 0.40]
  span         17/43 = 40%   [0.26, 0.54]
  strand       2/14 = 14%   [0.04, 0.40]

what SemCat said instead, on the columns it did not get right:
gt_facet     status       
function_id  unlabelled       24
             vacuous_label     5
             wrong_family      3
span         unlabelled       11
             vacuous_label     4
             wrong_family     11
strand       unlabelled       10
             vacuous_label     2


In [10]:
print("EXAMPLES: relevant columns carrying a vacuous or wrong-family label")
ex = rec[rec.status.isin(["vacuous_label", "wrong_family"])]
print(ex[["fq", "column_name", "gt_facet", "semcat_label", "status"]].head(25).to_string(index=False))

EXAMPLES: relevant columns carrying a vacuous or wrong-family label
                                                       fq           column_name    gt_facet    semcat_label        status
                  refdata.bvbrc.bvbrc_specialty_gene_fact        specialty_type function_id        category vacuous_label
                         refdata.bvbrc.bvbrc_feature_fact                strand      strand annotation_text vacuous_label
                       kbase.nmdc_arkin.metabolomics_gold            start_scan        span     measurement  wrong_family
                              nmdc.metadata.biosample_set              quad_pos        span           label vacuous_label
                                   kescience.mgnify.biome      has_kegg_pathway function_id            flag  wrong_family
        nmdc.metadata.collecting_biosamples_from_site_set              end_date        span       timestamp  wrong_family
                                     refdata.tcdb.go_xref               go_ter

### 2c. The pre-registered decision

Applying `RESEARCH_PLAN.md`'s decision rule to the two numbers above. This is arithmetic on
pre-committed thresholds, not interpretation.

### 2b-bis. Per-`schema_sig` rates (pre-registered)

`RESEARCH_PLAN.md` commits to computing **every rate twice** — once per column and once per
distinct `schema_sig` — because cloned schemas (NMDC's dated staging copies of
`genome_feature_set`, among others) share a labelling fate and would otherwise inflate *n*
without adding evidence. Where the two disagree, **the per-`schema_sig` figure is the one
quoted**.

A `schema_sig` group is scored as a single observation: correct only if every audited column
in it is correct, found only if every one is found. That is the conservative direction.

In [11]:
def dedup_rate(df, ok_mask, label):
    """Collapse rows sharing a (schema_sig, column_name) to one observation."""
    d = df.copy()
    d["_ok"] = ok_mask.values
    d["_sig"] = d.fq.map(SIG).fillna(d.fq)          # fall back to fq when sig is absent
    d["_key"] = d["_sig"].astype(str) + "::" + d.column_name.astype(str)
    grp = d.groupby("_key")["_ok"].min()             # all-must-pass within a clone group
    n_col, n_sig = len(d), len(grp)
    r_col, r_sig = d["_ok"].mean(), grp.mean()
    print(f"{label}")
    print(f"  per column      : {int(d['_ok'].sum())}/{n_col} = {r_col:.1%}")
    print(f"  per schema_sig  : {int(grp.sum())}/{n_sig} = {r_sig:.1%}")
    print(f"  collapsed       : {n_col - n_sig} duplicate observation(s)")
    return r_col, r_sig, n_sig

print("PRECISION (judgeable labels only)")
prec_sig = dedup_rate(checkable, (checkable.verdict == "correct"), "  strict precision")
print()
prec_len_sig = dedup_rate(checkable, checkable.verdict.isin(["correct", "wrong_cardinality"]),
                          "  lenient precision")
print()
print("RECALL (ground-truth columns)")
rec_sig = dedup_rate(rec, (rec.status == "found"), "  recall")

precision_strict_sig, precision_lenient_sig, recall_sig = prec_sig[1], prec_len_sig[1], rec_sig[1]
print()
print("Per the plan, the schema_sig figures are the ones quoted where they differ:")
print(f"  strict precision : {precision_strict_sig:.1%}   (per-column {precision_strict:.1%})")
print(f"  lenient precision: {precision_lenient_sig:.1%}   (per-column {precision_lenient:.1%})")
print(f"  recall           : {recall_sig:.1%}   (per-column {recall:.1%})")

PRECISION (judgeable labels only)
  strict precision
  per column      : 47/58 = 81.0%
  per schema_sig  : 45/56 = 80.4%
  collapsed       : 2 duplicate observation(s)

  lenient precision
  per column      : 51/58 = 87.9%
  per schema_sig  : 49/56 = 87.5%
  collapsed       : 2 duplicate observation(s)

RECALL (ground-truth columns)
  recall
  per column      : 30/100 = 30.0%
  per schema_sig  : 30/95 = 31.6%
  collapsed       : 5 duplicate observation(s)

Per the plan, the schema_sig figures are the ones quoted where they differ:
  strict precision : 80.4%   (per-column 81.0%)
  lenient precision: 87.5%   (per-column 87.9%)
  recall           : 31.6%   (per-column 30.0%)


In [12]:
# Evaluated on the per-schema_sig figures, as the plan requires.
print(f"audited recall (apt-family label)  : {recall_sig:.1%}   [per-column {recall:.1%}]")
print(f"audited precision, strict          : {precision_strict_sig:.1%}   [per-column {precision_strict:.1%}]")
print(f"audited precision, lenient         : {precision_lenient_sig:.1%}   [per-column {precision_lenient:.1%}]")
print()
recall, precision_strict, precision_lenient = recall_sig, precision_strict_sig, precision_lenient_sig
# 95% Wilson intervals on the quoted (per-schema_sig) figures.
def _k(rate, n): return int(round(rate * n))
ci_sig   = wilson(_k(precision_strict_sig, prec_sig[2]), prec_sig[2])
ci_sig_l = wilson(_k(precision_lenient_sig, prec_len_sig[2]), prec_len_sig[2])
ci_rec   = wilson(_k(recall_sig, rec_sig[2]), rec_sig[2])
print(f"strict precision  {_k(precision_strict_sig, prec_sig[2])}/{prec_sig[2]} = {precision_strict_sig:.1%}  "
      f"95% Wilson [{ci_sig[0]:.3f}, {ci_sig[1]:.3f}]")
print(f"lenient precision {_k(precision_lenient_sig, prec_len_sig[2])}/{prec_len_sig[2]} = {precision_lenient_sig:.1%}  "
      f"95% Wilson [{ci_sig_l[0]:.3f}, {ci_sig_l[1]:.3f}]")
print(f"recall            {_k(recall_sig, rec_sig[2])}/{rec_sig[2]} = {recall_sig:.1%}  "
      f"95% Wilson [{ci_rec[0]:.3f}, {ci_rec[1]:.3f}]")
print()
rc_ok  = bool(nb02["row_count_h3_clause_met"])   # measured live in NB02 §6a
cov_ok = nb01["candidates_in_inaccessible_tenants"] >= 1
h1 = (recall >= 0.80) and nb01["h1_coordinate_subtest_pass"]
h2_s = (recall < 0.50) and (precision_strict  >= 0.80)
h3_s = (recall < 0.50) and (precision_strict  <  0.80) and rc_ok and cov_ok
h2_l = (recall < 0.50) and (precision_lenient >= 0.80)
h3_l = (recall < 0.50) and (precision_lenient <  0.80) and rc_ok and cov_ok
print(f"H1  recall>=0.80 AND coordinate system resolvable -> {h1}"
      f"  (coordinate sub-test passed: {nb01['h1_coordinate_subtest_pass']})")
print(f"H2  strict -> {h2_s}    lenient -> {h2_l}")
print(f"H3  strict -> {h3_s}    lenient -> {h3_l}"
      f"   (row_count>=95% {rc_ok}; tables beyond own inventory: "
      f"{nb01['candidates_in_inaccessible_tenants']})")
print()
if h1:                     verdict = "H1 supported"
elif h2_s and h2_l:        verdict = "H2 supported"
elif h3_s and h3_l:        verdict = "H3 supported"
else:
    verdict = ("MIXED -- recall decisively refutes H1, but precision straddles the 0.80 "
               "H2/H3 boundary depending on whether multi-valued columns count as "
               "mislabelled; this sample does not resolve H2 vs H3")
print("PRE-REGISTERED VERDICT:", verdict)
print()
print(f"Uncertainty: strict precision 95% Wilson interval [{ci_sig[0]:.3f}, {ci_sig[1]:.3f}] "
      f"{'lies above' if ci_sig[0] >= 0.80 else 'includes values below'} the 0.80 H2/H3 boundary; "
      f"lenient [{ci_sig_l[0]:.3f}, {ci_sig_l[1]:.3f}] "
      f"{'lies above' if ci_sig_l[0] >= 0.80 else 'includes values below'} it.")
print("The pre-registered rule compares point estimates; when the interval straddles 0.80 the")
print("sample does not separate H2 from H3 at 95% confidence, whatever the point verdict says.")

audited recall (apt-family label)  : 31.6%   [per-column 30.0%]
audited precision, strict          : 80.4%   [per-column 81.0%]
audited precision, lenient         : 87.5%   [per-column 87.9%]

strict precision  45/56 = 80.4%  95% Wilson [0.682, 0.887]
lenient precision 49/56 = 87.5%  95% Wilson [0.764, 0.938]
recall            30/95 = 31.6%  95% Wilson [0.231, 0.415]

H1  recall>=0.80 AND coordinate system resolvable -> False  (coordinate sub-test passed: False)
H2  strict -> True    lenient -> True
H3  strict -> False    lenient -> False   (row_count>=95% True; tables beyond own inventory: 483)

PRE-REGISTERED VERDICT: H2 supported

Uncertainty: strict precision 95% Wilson interval [0.682, 0.887] includes values below the 0.80 H2/H3 boundary; lenient [0.764, 0.938] includes values below it.
The pre-registered rule compares point estimates; when the interval straddles 0.80 the
sample does not separate H2 from H3 at 95% confidence, whatever the point verdict says.


## 3. The H0 test — does SemCat beat a name regex?

NB01 counted the tables flagged by SemCat's *specific* labels that a name regex over the same
column inventory missed (`only_route_a_specific`). H0 (no added value) is rejected only if some of those are **true
positives** — genuinely relevant tables a name-based search would not find.

In [13]:
only_a = cand[(cand.route_a_specific == 1) & (cand.route_b_candidate == 0)]
only_a_acc = only_a[only_a.tenant.isin(TENANTS)]
print(f"SemCat-specific-only candidate tables: {len(only_a)} total, "
      f"{len(only_a_acc)} in accessible tenants")

# Which of those are in the adjudicated census, i.e. genuinely relevant?
cen_fqs = set(cen.fq)
hit = only_a_acc[only_a_acc.fq.isin(cen_fqs)]
print(f"of those, confirmed relevant by the census: {len(hit)}")
if len(hit):
    print(hit[["fq", "row_count"]].to_string(index=False))

only_b = cand[(cand.route_a_specific == 0) & (cand.route_b_candidate == 1)]
only_b_acc = only_b[only_b.tenant.isin(TENANTS)]
hit_b = only_b_acc[only_b_acc.fq.isin(cen_fqs)]
print()
print(f"regex-only candidate tables: {len(only_b)} total, {len(only_b_acc)} accessible, "
      f"{len(hit_b)} confirmed relevant")
if len(hit_b): print(hit_b[["fq", "row_count"]].to_string(index=False))

SemCat-specific-only candidate tables: 391 total, 26 in accessible tenants
of those, confirmed relevant by the census: 2
                                                         fq  row_count
globalusers.carbon_source_phenotypes.kofam_annotation_table    5235819
                  kbase.nmdc_arkin.metatranscriptomics_gold   75119498

regex-only candidate tables: 482 total, 150 accessible, 11 confirmed relevant
                                           fq  row_count
globalusers.gapmind_pathways.gapmind_pathways  463729001
                        kbase.genomes.feature 1011650903
          kbase.ke_pangenome.gapmind_pathways  305471280
      kescience.fitnessbrowser.seedannotation     177519
                    kescience.mgnify.gene_amr       8862
    nmdc.results.annotation_enzyme_commission 1231453377
                     refdata.cazy.annotations    6006576
        refdata.fungal_db.pombase_gene_coords      12685
                refdata.jgi_mycocosm.features   17945606
           refdat

In [14]:
# Which route would have found each census table? This is the practically useful answer.
cenj = cen.merge(cand[["fq", "route_a_specific", "route_a_candidate", "route_b_candidate",
                       "a_strand", "a_span", "a_func_specific"]], on="fq", how="left")
cenj["in_semcat"] = cenj.route_a_candidate.notna()
def route_verdict(r):
    if not r.in_semcat:                      return "missed_absent_from_semcat"
    if r.route_a_specific == 1:              return "found_by_specific_label"
    if r.route_a_candidate == 1:             return "found_by_generic_label_only"
    return "missed_unlabelled"
cenj["semcat_verdict"] = cenj.apply(route_verdict, axis=1)
print("How SemCat fares on the 51 census tables:")
print(cenj.semcat_verdict.value_counts().to_string())
print()
print(cenj[["fq", "category", "coord_system", "semcat_verdict"]].sort_values(
    ["semcat_verdict", "fq"]).to_string(index=False))

How SemCat fares on the 51 census tables:


semcat_verdict
found_by_specific_label        37
found_by_generic_label_only    13
missed_absent_from_semcat       1

                                                         fq             category coord_system              semcat_verdict
              globalusers.gapmind_pathways.gapmind_pathways        gene_function         none found_by_generic_label_only
                                      kbase.genomes.feature               region   nucleotide found_by_generic_label_only
                        kbase.ke_pangenome.gapmind_pathways        gene_function         none found_by_generic_label_only
                    kescience.fitnessbrowser.seedannotation        gene_function         none found_by_generic_label_only
                                  kescience.mgnify.gene_amr        gene_function         none found_by_generic_label_only
                  nmdc.results.annotation_enzyme_commission                 both     multiple found_by_generic_label_only
                    refdata.

In [15]:
cenj[["fq", "category", "region_type", "coords", "coord_system", "coord_frame", "seq_ref",
      "span", "strand", "id_systems", "row_count", "semcat_verdict"]].to_csv(
    "../data/semcat_grade.csv", index=False)
prec.to_csv("../data/audit_precision.csv", index=False)
rec.to_csv("../data/audit_recall.csv", index=False)
print("wrote ../data/semcat_grade.csv, audit_precision.csv, audit_recall.csv")

wrote ../data/semcat_grade.csv, audit_precision.csv, audit_recall.csv


## 4. Per-facet confusion over the whole frame

The 200-column audit is the pre-registered test; this is the full-frame view, computed by rule
rather than by value-adjudication, as a consistency check. If the two disagree materially, the
audit is the one that counts (it looked at data; this looks at names).

In [16]:
conf = []
for fam, labs in FAMILY.items():
    if fam == "seqref": continue
    t_nodes = set(truth[truth.gt_facet == fam].node)
    s_nodes = set(scf[scf.content_type.isin(labs)].node)
    tp, fp, fn_ = len(t_nodes & s_nodes), len(s_nodes - t_nodes), len(t_nodes - s_nodes)
    conf.append(dict(facet=fam, ground_truth=len(t_nodes), semcat_labelled=len(s_nodes),
                     tp=tp, fp=fp, fn=fn_,
                     recall=tp/(tp+fn_) if tp+fn_ else float("nan"),
                     precision=tp/(tp+fp) if tp+fp else float("nan")))
cf = pd.DataFrame(conf)
print("FULL-FRAME, RULE-BASED (biased against SemCat: the name regex is treated as truth, so a")
print("correct label on a column the regex missed counts as a false positive). Not a precision estimate.")
print(cf.to_string(index=False))
cf.to_csv("../data/semcat_confusion_by_facet.csv", index=False)

FULL-FRAME, RULE-BASED (biased against SemCat: the name regex is treated as truth, so a
correct label on a column the regex missed counts as a false positive). Not a precision estimate.
      facet  ground_truth  semcat_labelled  tp  fp  fn   recall  precision
function_id           621              277 147 130 474 0.236715   0.530686
       span            88               87  38  49  50 0.431818   0.436782
     strand            14                4   2   2  12 0.142857   0.500000


In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Both panels use the value-adjudicated audit only. The full-frame rule-based rates (section 4)
# are biased against SemCat and are not plotted.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
facets = ["function_id", "span", "strand", "seqref"]
x = np.arange(len(facets)); w = 0.38
def bars(d, key, offset, colour, label):
    vals = [d[f]["rate"] if f in d else np.nan for f in facets]
    lo = [d[f]["rate"] - d[f]["ci_low"] if f in d else 0 for f in facets]
    hi = [d[f]["ci_high"] - d[f]["rate"] if f in d else 0 for f in facets]
    ax.bar(x + offset, np.nan_to_num(vals), w, yerr=[lo, hi], capsize=3, color=colour, label=label)
    for xi, f in zip(x + offset, facets):
        if f in d:
            ax.text(xi, 0.02, f"{d[f][key]}/{d[f]['judged' if key == 'correct' else 'n']}",
                    ha="center", fontsize=8, color="white", fontweight="bold")
bars(fac_rec, "found", -w/2, "#c44e52", "recall (audit sample)")
bars(fam_prec, "correct", +w/2, "#4c72b0", "strict precision (audit, judgeable labels)")
ax.axhline(0.80, ls="--", c="k", lw=1)
ax.text(len(facets) - 0.5, 0.815, "H1 recall / H2 precision threshold = 0.80", ha="right", fontsize=8)
ax.axhline(0.50, ls=":", c="gray", lw=1)
ax.text(len(facets) - 0.5, 0.515, "H2/H3 recall ceiling = 0.50", ha="right", fontsize=8, color="gray")
ax.set_xticks(x); ax.set_xticklabels(facets); ax.set_ylim(0, 1.12)
ax.set_ylabel("rate (per column), 95% Wilson interval")
ax.set_title("SemCat label quality by facet\n(value-adjudicated audit; seqref has no recall stratum)")
ax.legend(loc="upper left", fontsize=8)

ax = axes[1]
pv = prec.verdict.value_counts()
order = [k for k in ["correct", "vacuous", "wrong_cardinality", "wrong_label",
                     "sentinel_only", "no_pattern", "no_data"] if k in pv]
cols = {"correct": "#55a868", "vacuous": "#bbbbbb", "wrong_cardinality": "#dd8452",
        "wrong_label": "#c44e52", "sentinel_only": "#8172b3", "no_pattern": "#937860",
        "no_data": "#cccccc"}
ax.bar(order, [pv[k] for k in order], color=[cols[k] for k in order])
for i, k in enumerate(order): ax.text(i, pv[k] + 1, str(pv[k]), ha="center", fontsize=9)
ax.set_ylabel("columns")
ax.set_title(f"Audit of {len(prec)} SemCat-labelled columns (per schema_sig): strict "
             f"{precision_strict:.1%} [{ci_sig[0]:.0%}, {ci_sig[1]:.0%}] / lenient {precision_lenient:.1%}")
ax.tick_params(axis="x", rotation=30)
for _l in ax.get_xticklabels(): _l.set_ha("right")
plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/semcat_precision_recall.png", dpi=150, bbox_inches="tight")
print("wrote ../figures/semcat_precision_recall.png")
plt.show()

wrote ../figures/semcat_precision_recall.png


In [18]:
summary = dict(
    audited_frame_tables=len(frame_tbls),
    in_frame_semcat_columns=int(len(scf)),
    content_type_populated_frac=float(scf.content_type.notna().mean()),
    ground_truth_relevant_columns=int(len(truth)),
    audit_precision_sample=int(len(prec)),
    value_sampling='DISTINCT + ORDER BY xxhash64(value), hash-bucketed where ndv>100000',
    audit_checkable=int(len(checkable)),
    audit_correct=n_corr, audit_wrong_cardinality=n_card, audit_wrong_label=n_wrong,
    audit_vacuous=int((prec.verdict == "vacuous").sum()),
    audit_sentinel_only=int((prec.verdict == "sentinel_only").sum()),
    audited_precision_strict=float(precision_strict_sig),
    audited_precision_lenient=float(precision_lenient_sig),
    audited_precision_strict_per_column=float(prec_sig[0]),
    audited_precision_lenient_per_column=float(prec_len_sig[0]),
    schema_sig_groups_precision=int(prec_sig[2]),
    audit_recall_sample=int(len(rec)),
    audit_found=int((rec.status == "found").sum()),
    audited_recall=float(recall_sig),
    audited_recall_per_column=float(rec_sig[0]),
    schema_sig_groups_recall=int(rec_sig[2]),
    verdict=verdict,
    census_found_by_specific_label=int((cenj.semcat_verdict == "found_by_specific_label").sum()),
    census_found_by_generic_only=int((cenj.semcat_verdict == "found_by_generic_label_only").sum()),
    census_missed_unlabelled=int((cenj.semcat_verdict == "missed_unlabelled").sum()),
    census_missed_absent=int((cenj.semcat_verdict == "missed_absent_from_semcat").sum()),
    h0_semcat_only_true_positives=int(len(hit)),
    h0_regex_only_true_positives=int(len(hit_b)),
    h0_semcat_only_true_positive_tables=hit.fq.tolist(),
    audit_no_data=int((prec.verdict == "no_data").sum()),
    audit_no_pattern=int((prec.verdict == "no_pattern").sum()),
    values_query_failed=int(errs), values_empty=int(empty),
    precision_strict_sig_ci95_wilson=[round(ci_sig[0], 4), round(ci_sig[1], 4)],
    precision_lenient_sig_ci95_wilson=[round(ci_sig_l[0], 4), round(ci_sig_l[1], 4)],
    recall_sig_ci95_wilson=[round(ci_rec[0], 4), round(ci_rec[1], 4)],
    precision_strict_sig_correct=_k(precision_strict_sig, prec_sig[2]),
    recall_sig_found=_k(recall_sig, rec_sig[2]),
    precision_strict_ci95_wilson=[round(ci_s[0], 4), round(ci_s[1], 4)],
    precision_strict_ci95_normal=[round(wald_s[0], 4), round(wald_s[1], 4)],
    precision_lenient_ci95_wilson=[round(ci_l[0], 4), round(ci_l[1], 4)],
    precision_by_family={k: {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in v.items()}
                         for k, v in fam_prec.items()},
    recall_by_facet={k: {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in v.items()}
                     for k, v in fac_rec.items()},
    recall_found_per_column=int((rec.status == "found").sum()),
    full_frame_function_id_semcat_only_columns=int(cf.loc[cf.facet == "function_id", "fp"].iloc[0]),
    row_count_clause_from_nb02=rc_ok,
    notebook_seconds=round(time.time() - _T0),
)
bad = {k: type(v).__name__ for k, v in summary.items()
       if not isinstance(v, (int, float, str, bool, list, dict, type(None)))}
if bad: print("NON-SERIALIZABLE:", bad)
json.dump(summary, open("../data/nb03_summary.json", "w"), indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))
print(f"NB03 wall time: {summary['notebook_seconds']} s")

{
  "audited_frame_tables": 655,
  "in_frame_semcat_columns": 66374,
  "content_type_populated_frac": 0.0822309940639407,
  "ground_truth_relevant_columns": 723,
  "audit_precision_sample": 100,
  "value_sampling": "DISTINCT + ORDER BY xxhash64(value), hash-bucketed where ndv>100000",
  "audit_checkable": 58,
  "audit_correct": 47,
  "audit_wrong_cardinality": 4,
  "audit_wrong_label": 7,
  "audit_vacuous": 17,
  "audit_sentinel_only": 1,
  "audited_precision_strict": 0.8035714285714286,
  "audited_precision_lenient": 0.875,
  "audited_precision_strict_per_column": 0.8103448275862069,
  "audited_precision_lenient_per_column": 0.8793103448275862,
  "schema_sig_groups_precision": 56,
  "audit_recall_sample": 100,
  "audit_found": 30,
  "audited_recall": 0.3157894736842105,
  "audited_recall_per_column": 0.3,
  "schema_sig_groups_recall": 95,
  "verdict": "H2 supported",
  "census_found_by_specific_label": 37,
  "census_found_by_generic_only": 13,
  "census_missed_unlabelled": 0,
  "censu